# P99 — Localización y mapeo simultáneos: parte I

## 1. Título y paper

**Paper:** *Simultaneous Localization and Mapping: Part I*  
**Autoría:** Hugh Durrant-Whyte, Tim Bailey  
**Año y venue:** 2006 · IEEE Robotics & Automation Magazine, 13(2), 99–110  
**Nivel:** L3 · **Motor:** `slam`  
**Ficha completa:** [`P99_slam`](../../papers/foundational/P99_slam/README.md)

**Hito:** Formaliza el problema circular de la robótica móvil: no se puede localizar sin mapa ni mapear sin localización, y hay que resolver ambos a la vez.

- [doi:10.1109/MRA.2006.1638022](https://doi.org/10.1109/MRA.2006.1638022)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un robot que se mueve acumula error de odometría sin límite. Corregirlo exige referencias externas; pero si el mapa no existe de antemano, hay que construirlo con la misma pose incierta que se quiere corregir.
2. Ejecutar una implementación mínima de la propuesta: Estimar el estado conjunto —pose y mapa— reconociendo que sus errores están **correlacionados**. El artículo formaliza la estructura de la covarianza, explica por qué converge y por qué el cierre de bucle corrige la trayectoria entera.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P96
- Smith, Self y Cheeseman (1990), relaciones espaciales inciertas


## 4. Intuición

Para saber dónde estás necesitas un mapa. Para hacer un mapa necesitas saber dónde estás. No es un juego de palabras: es la estructura del problema, y la única salida es estimar las dos cosas a la vez aceptando que sus errores están enredados.


## 5. Concepto mínimo

```text
Odometría sola:     la varianza de la pose CRECE sin techo
Con balizas:        cada reencuentro reduce la varianza

La baliza se sitúa usando la pose  →  su error hereda el de la pose
La pose se corrige usando la baliza →  su error hereda el de la baliza

⟹ hay que estimar el estado CONJUNTO, con su covarianza cruzada
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('slam', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Hasta dónde crece el error con odometría sola?
2. ¿Y estimando pose y mapa a la vez?
3. ¿Qué aporta reencontrar una baliza vista al principio?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('slam', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('slam', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Solo con odometría el error llega a **1,227** y su varianza a **10,8**: crece sin techo porque nada la corrige. Estimando pose y mapa a la vez baja a **1,0311** con varianza 5,7 — y las balizas no se conocían de antemano. Al cerrar el bucle, el error final cae a **0,5043**.


## 10. Comentario pedagógico

El cierre de bucle es el evento más valioso de un recorrido y por eso los sistemas reales invierten tanto en detectarlo: reconocer un sitio ya visitado corrige de golpe toda la deriva acumulada desde entonces. Su contrapartida es que equivocarse al reconocerlo —creer que estás en un sitio donde no estás— destruye el mapa entero.


## 11. Error o anti-patrón deliberado

Anti-patrón: tratar la asociación de datos como un detalle de implementación.


In [ ]:
print('El problema DIFICIL de SLAM no es el filtro: es decidir si esta baliza')
print('es la misma que vi hace diez minutos. Se llama asociacion de datos.')
print('Un falso positivo ahi no degrada el mapa: lo rompe, y sin aviso.')

## 12. Corrección

Lo que la miniatura sí demuestra:


In [ ]:
r = run_paper_lab('slam', seed=7)['result']
print('solo odometria    :', r['solo_odometria']['error_pose_final'],
      '| varianza', r['solo_odometria']['varianza_pose_final'])
print('SLAM sin cierre   :', r['slam_sin_cierre_de_bucle']['error_pose_final'])
print('SLAM con cierre   :', r['slam_con_cierre_de_bucle']['error_pose_final'])
print('mapa estimado     :', r['slam_con_cierre_de_bucle']['mapa'])

## 13. Desafío guiado

Sigue la curva de varianza de la pose y localiza en qué pasos baja: coinciden con los reencuentros de balizas.


In [ ]:
r = run_paper_lab('slam', seed=3)['result']
show(r)

## 14. Desafío autónomo

Implementa SLAM en dos dimensiones con balizas indistinguibles entre sí, y comprueba qué pasa cuando el sistema asocia mal dos de ellas. Documenta el efecto sobre el mapa completo.


## 15. Evidencia de aprendizaje

Guarda la comparación de los tres escenarios y tu explicación de por qué los errores de pose y mapa están correlacionados.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P99_slam/README.md) · evaluación formal: [`assessments/papers/P99_slam.md`](../../assessments/papers/P99_slam.md)


## 16. Cierre

El robot ya sabe dónde está y sabe llegar. Falta lo que hace mientras se mueve: el control, y si se escribe o se aprende.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
